In [0]:
!pip install xarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import io
import os
import re
import json
import uuid
import zipfile
import tempfile
import xarray as xr


def sanitize_column_name(col_name: str) -> str:
    c = col_name.strip().lower()
    c = re.sub(r"[^a-z0-9_]", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    if not c:
        c = "col"
    if c[0].isdigit():
        c = f"col_{c}"
    return c


def sanitize_pandas_columns(pdf: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    seen = set()
    mapping = {}
    new_cols = []

    for c in pdf.columns:
        safe = sanitize_column_name(str(c))
        base = safe
        i = 2
        while safe in seen:
            safe = f"{base}_{i}"
            i += 1
        seen.add(safe)
        mapping[c] = safe
        new_cols.append(safe)

    out = pdf.copy()
    out.columns = new_cols
    return out, mapping


def load_binary_bytes(path: str) -> bytes:
    row = (
        spark.read.format("binaryFile")
        .load(path)
        .select("content")
        .first()
    )
    return bytes(row["content"])


def load_binary_to_tempfile(path: str, suffix: str = "") -> str:
    raw = load_binary_bytes(path)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(raw)
    tmp.flush()
    tmp.close()
    return tmp.name


def write_pandas_to_table(pdf: pd.DataFrame, table_name: str, mode: str = "append"):
    if pdf.empty:
        return

    pdf = pdf.where(pd.notnull(pdf), None)
    sdf = spark.createDataFrame(pdf)
    sdf.write.mode(mode).saveAsTable(table_name)


def write_pandas_csv_chunks_from_zip_member(
    zip_bytes: bytes,
    member_name: str,
    table_name: str,
    chunksize: int = 200_000,
):
    first = True
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        with zf.open(member_name) as fh:
            reader = pd.read_csv(
                fh,
                chunksize=chunksize,
                encoding="cp1252",
                low_memory=False,
            )
            for chunk in reader:
                chunk, _ = sanitize_pandas_columns(chunk)
                chunk["source_member_name"] = member_name
                chunk["source_zip_name"] = "CalCOFI_Database_194903-202105_csv_16October2023.zip"
                write_pandas_to_table(
                    chunk,
                    table_name,
                    mode="overwrite" if first else "append",
                )
                first = False

In [0]:
raw_inventory = spark.table("toxictide.bronze.raw_file_inventory")
raw_inventory_summary = spark.table("toxictide.bronze.raw_file_inventory_summary")

display(raw_inventory_summary.orderBy("source_name"))
display(raw_inventory.limit(20))

source_name,n_files,total_bytes,min_bytes,max_bytes
ca_beach_water_quality,3,1786512559,649554,1755348817
calcofi,2,37782792,3564,37779228
cce_moorings,266,997160828,8192,84858112
easyoneargo,4,19058531730,1586383338,7969262537


source_name,path,length,modificationTime
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_D_Meteorology-Wind.nc,84858112,2026-04-19T03:42:44.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_D_NO3.nc,143136,2026-04-19T03:40:36.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_D_OXYGEN.nc,160160,2026-04-19T03:40:38.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_P_PH-40m.nc,136764,2026-04-19T03:40:40.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_ADCP.nc,7475608,2026-04-19T03:41:26.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_CHL.nc,9844,2026-04-19T03:40:49.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_CTD.nc,2318212,2026-04-19T03:41:04.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_Meteorology-Rain.nc,436784,2026-04-19T03:40:59.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_Meteorology-TempHumiPres.nc,896408,2026-04-19T03:41:15.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_12_D_Meteorology-Wind.nc,14360988,2026-04-19T03:42:11.000Z


# CalCOFI

In [0]:
calcofi_inventory = (
    raw_inventory
    .filter(F.col("source_name") == "calcofi")
    .withColumn("filename", F.regexp_extract("path", r"/([^/]+)$", 1))
)

display(calcofi_inventory)

source_name,path,length,modificationTime,filename
calcofi,s3://toxictide-raw/sources/calcofi/raw/run_date=2026-04-19/bottle_database/CalCOFI_Database_194903-202105_csv_16October2023.zip,37779228,2026-04-19T08:02:34.000Z,CalCOFI_Database_194903-202105_csv_16October2023.zip
calcofi,s3://toxictide-raw/sources/calcofi/raw/run_date=2026-04-19/ctd_cast_files/Cast_Field_Descriptions.csv,3564,2026-04-19T08:02:34.000Z,Cast_Field_Descriptions.csv


In [0]:
calcofi_cast_fields = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("s3://toxictide-raw/sources/calcofi/raw/*/ctd_cast_files/*.csv")
)

display(calcofi_cast_fields.limit(50))
print("Columns:", calcofi_cast_fields.columns)

calcofi_cast_fields_clean = calcofi_cast_fields.toDF(
    *[sanitize_column_name(c) for c in calcofi_cast_fields.columns]
)

calcofi_cast_fields_clean.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.calcofi_cast_field_descriptions_raw"
)

Field Name,Units,Description
Cst_Cnt,n.a.,"Cast Count - All CalCOFI casts ever conducted, consecutively numbered"
Cruise_ID,n.a.,Cruise identifier [Year]-[Month]-[Day]-C-[Ship Code]
Cruise,n.a.,Cruise Name [Year][Month]
Cruz_Sta,n.a.,Cruise Name and Station [Year][Month][Line][Station]
DbSta_ID,n.a.,Line and Station
Cast_ID,n.a.,Cast Identifier [Century] - [YY][MM][ShipCode] - [CastType][Julian Day] - [CastTime]-[Line][Sta]
Sta_ID,n.a.,Line and Station
Quarter,n.a.,Quarter of the year
Sta_Code,n.a.,Station Designation (See Station_ID and 0-St_Code for codes)
Distance,nautical miles,"Nautical miles from coast intercept, calculated from estimated latitude and longitude"


Columns: ['Field Name', 'Units', 'Description']


In [0]:
calcofi_zip_path = (
    calcofi_inventory
    .filter(F.col("filename").endswith(".zip"))
    .select("path")
    .first()["path"]
)

zip_bytes = load_binary_bytes(calcofi_zip_path)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    zip_members = [
        {
            "member_name": zi.filename,
            "file_size": zi.file_size,
            "compressed_size": zi.compress_size,
            "is_dir": zi.is_dir(),
        }
        for zi in zf.infolist()
    ]

calcofi_zip_members_df = spark.createDataFrame(pd.DataFrame(zip_members))
display(calcofi_zip_members_df.orderBy(F.desc("file_size")))

calcofi_zip_members_df.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.calcofi_zip_member_inventory"
)

member_name,file_size,compressed_size,is_dir
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Bottle.csv,175418694,35325993,false
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Cast.csv,12525596,2452305,false
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/,0,0,true


In [0]:
csv_members = [r["member_name"] for r in zip_members if r["member_name"].lower().endswith(".csv")]
print("CSV members inside zip:")
for m in csv_members:
    print(" -", m)

CSV members inside zip:
 - CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Bottle.csv
 - CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Cast.csv


In [0]:
for member_name in csv_members:
    table_suffix = sanitize_column_name(member_name.replace(".csv", ""))
    table_name = f"toxictide.bronze.calcofi_{table_suffix}_raw"

    print(f"Parsing {member_name} -> {table_name}")
    write_pandas_csv_chunks_from_zip_member(
        zip_bytes=zip_bytes,
        member_name=member_name,
        table_name=table_name,
        chunksize=200_000,
    )

Parsing CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Bottle.csv -> toxictide.bronze.calcofi_calcofi_database_194903_202105_csv_16october2023_calcofi_database_194903_202105_csv_16october2023_194903_202105_bottle_raw


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6904815970628422>, line 6
      3 table_name = f"toxictide.bronze.calcofi_{table_suffix}_raw"
      5 print(f"Parsing {member_name} -> {table_name}")
----> 6 write_pandas_csv_chunks_from_zip_member(
      7     zip_bytes=zip_bytes,
      8     member_name=member_name,
      9     table_name=table_name,
     10     chunksize=200_000,
     11 )

File <command-6904815970628413>, line 93, in write_pandas_csv_chunks_from_zip_member(zip_bytes, member_name, table_name, chunksize)
     91 chunk["source_member_name"] = member_name
     92 chunk["source_zip_name"] = "CalCOFI_Database_194903-202105_csv_16October2023.zip"
---> 93 write_pandas_to_table(
     94     chunk,
     95     table_name,
     96     mode="overwrite" if first else "append",
     97 )
     98 first = False

File <command-6904815970628413>, line 71, in write_panda

In [0]:
display(spark.table("toxictide.bronze.calcofi_cast_field_descriptions_raw").limit(20))

for member_name in csv_members:
    table_suffix = sanitize_column_name(member_name.replace(".csv", ""))
    table_name = f"toxictide.bronze.calcofi_{table_suffix}_raw"
    print(table_name, spark.table(table_name).count())

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

# CA Beach

In [0]:
ca_tables = [
    "toxictide.bronze.ca_beach_monitoring_stations_raw",
    "toxictide.bronze.ca_beach_postings_closures_raw",
    "toxictide.bronze.ca_beach_monitoring_results_raw",
]

for t in ca_tables:
    print(f"\n=== {t} ===")
    df = spark.table(t)
    print("rows =", df.count(), "cols =", len(df.columns))
    display(df.limit(10))


=== toxictide.bronze.ca_beach_monitoring_stations_raw ===
rows = 1075 cols = 61


regional_board,station_id,station_agency_id,beachname_id,station_name,station_description,agencystationidentifier,station_upperlat,station_upperlon,station_lowerlat,station_lowerlon,swimseason,offseason,asneededseason,countycode,countyname,status,updatedate,stations_insert_date,pointzero,datum,beach_agencyname,beach_name,description,beachtype,beach_length,attendancewinter,attendancesummer,nearestcityname,waterbodyname,waterbodyclass,ab411beach,usepaid,beachaccess,swimseasonlength,waterbodytype,watershedname,beach_upperlat,beach_upperlon,beach_lowerlat,beach_lowerlon,beach_county,beach_user_id,beach_status,countasbeach,agency_id,agency_code,agency_name,agency_jurisdiction,streetaddress,city,station_contact_person,zip,estmoncost,estcoastlineadvisorycost,esttotalcost,station_comments,county,agency_user_id,regional_board_number,regional_board_name
RB8,1,21,53,0,50\' N of Santa Ana River,0,33.62929,-117.95968,33.62929,-117.95968,Twice a week,Twice a week,Once per day,OC,Orange,Active,2026-02-09,InsertDate,1,4,Orange County Environmental Health Division,Huntington State Beach,Huntington State Beach,UNKNOWN,2.01,0,0,Huntington Beach,Pacific Ocean,Saltwater,No,CA857004,PUBLIC,12,Open Coast,Seal Beach,33.646021,-117.988024,33.629289,-117.959471,Orange,3,Active,1,21,EH,Orange County Environmental Health Division,County,2009 E. Edinger Avenue,Santa Ana,Michael Fennessy,92705,NULL,NULL,More than $250000,Michael Fennessy,Orange,1,8,Santa Ana
RB3,2,32,145,1000,Rincon Beach,1000,34.373398,-119.476088,34.373287201,-119.35891,Once a week,Once a week,Twice a week,VC,Ventura,Active,2015-08-25,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB4,3,32,148,10000,Solimar Beach,10000,34.310402,-119.35891,34.310402,-119.476524353,Once a week,Once a week,Twice a week,VC,Ventura,Active,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Solimar Beach,Solimar Beach,UNKNOWN,1.57,0,0,Solimar Beach,Pacific Ocean,Saltwater,No,CA844317,PUBLIC,12,Open Coast,Ventura River,34.32101,-119.37541,34.307619,-119.353167,Ventura,7,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,4,Los Angeles
RB3,4,32,145,1001,Rincon Creek (mouth),1001,34.37351,-119.4756317,34.37351,-119.4756317,None(not sampled),None(not sampled),None(not sampled),VC,Ventura,Inactive,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB3,5,32,145,1050,Rincon Beach,1050,34.3739,-119.4749451,34.3739,-119.4749451,None(not sampled),None(not sampled),None(not sampled),VC,Ventura,Inactive,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB3,6,32,145,1100,Rincon Beach,1100,34.37538,-119.4730453,34.37538,-119.4730453,Once a week,None(not sampled),None(not sampled),VC,Ventura,Active,2015-09-09,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon B


=== toxictide.bronze.ca_beach_postings_closures_raw ===
rows = 36765 cols = 98


duration_calculated,regional_board,advisory_id,historicalstationname,prefix,mile,dateofadvisory,timeofadvisory,dateopened,timeopened,advisorytype,advisorycause,description,extent,startofextent,startofextenttype,endofextent,endofextenttype,source,advisory_comments,descriptionofarea,actionby,affectedlengthunit,substancename,substancevolume,substanceunit,additionalaction,actiondate,actiondescription,county,user_id,enterococcus,fecal_coliforms,total_coliforms,e_coli,ratio,advisory_record_create_date,station_id,station_agency_id,beachname_id,station_name,station_description,agencystationidentifier,station_upperlat,station_upperlon,station_lowerlat,station_lowerlon,swimseason,offseason,asneededseason,countycode,countyname,status,updatedate,advisories_insert_date,pointzero,datum,beachprofileid,beach_agencyname,beach_name,beachdescription,beachtype,beach_length,attendancewinter,attendancesummer,nearestcityname,waterbodyname,waterbodyclass,ab411beach,usepaid,beachaccess,swimseasonlength,waterbodytype,watershedname,beach_upperlat,beach_upperlon,beach_lowerlat,beach_lowerlon,beach_county,beach_user_id,beach_status,countasbeach,agency_id,agency_code,agency_name,agency_jurisdiction,streetaddress,city,advisories_contact_person,zip,estmoncost,estcoastlineadvisorycost,esttotalcost,agency_comments,agency_county,agency_user_id,regional_board_number,regional_board_name
1,RB4,433,B-24,I,0,5/3/2010,14:35:33,5/4/2010,14:37:02,Posting,Unknown Cause,Colorado Lagoon-South,0.05681818,-0.02840909,Lat/Long,0.02840909,Lat/Long,Unknown,null,Long Beach Colorado Lagoon-South,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,1,1,0,1,9/19/2015 3:39:59 PM,110,4,455,B-24,Colorado Lagoon-South,B-24,33.77062,-118.13475,33.77062,-118.13475,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,455,City of Long Beach Health and Human Services,Colorado Lagoon,Colorado Lagoon,ROCKY,0.1,0,0,LB,Pacific Ocean,Estuarine,No,CA699607,PUBLIC,12,"Sound, Bay, or Inlet",San Gabriel River,33.771323,-118.133145,33.770194,-118.132057,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
31,RB4,434,B-25,I,0,4/3/2010,14:35:51,5/4/2010,14:37:12,Posting,Unknown Cause,Colorado Lagoon-North,0.05681818,0.02249091,Lat/Long,0.07930909,Lat/Long,Unknown,null,Long Beach Colorado Lagoon-North,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,null,null,null,null,9/19/2015 3:39:59 PM,111,4,455,B-25,Colorado Lagoon-North,B-25,33.7712,-118.13435,33.7712,-118.13435,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,455,City of Long Beach Health and Human Services,Colorado Lagoon,Colorado Lagoon,ROCKY,0.1,0,0,LB,Pacific Ocean,Estuarine,No,CA699607,PUBLIC,12,"Sound, Bay, or Inlet",San Gabriel River,33.771323,-118.133145,33.770194,-118.132057,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
1,RB4,435,B-5,C,3,5/17/2010,14:47:43,5/18/2010,15:06:03,Posting,Unknown Cause,5th Place-Beach,0.05681818,3.799591,Lat/Long,3.856409,Lat/Long,Unknown,null,Long Beach 5th Place-Beach,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,null,null,null,null,9/19/2015 3:39:59 PM,125,4,381,B-5,5th Place-Beach,B-5,33.76393,-118.1784286,33.76393,-118.1784286,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,381,City of Long Beach Health and Human Services,Long Beach,Long Beach,UNKNOWN,3.7,0,0,LB,Pacific Ocean,Saltwater,No,CA279698,PUBLIC,12,Open Coast,San Gabriel River,33.74515,-118.119183,33.763467,-118.179767,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
1,RB4,436,B-


=== toxictide.bronze.ca_beach_monitoring_results_raw ===
rows = 2392908 cols = 93


regional_board,results_id,sampledate,starttime,starttimeind,parameter,qualifierid,qualifier,qualifierdescription,result,unit,labcode_id,analysismethodid,analysismethod,sampletypeid,sampletype,results_comments,county,user_id,upload_ts,result_record_create_date,labid,stormdrainflow,weather,tidalheight,surfheight,turbidity,watercolor,odor,distance,direction,rl,mdl,dilution,station_id,station_agency_id,beachname_id,station_name,station_description,agencystationidentifier,station_upperlat,station_upperlon,station_lowerlat,station_lowerlon,swimseason,offseason,asneededseason,countycode,countyname,status,updatedate,results_insert_date,pointzero,beach_agencyname,beach_name,description,beachtype,beach_length,attendancewinter,attendancesummer,nearestcityname,waterbodyname,waterbodyclass,ab411beach,usepaid,beachaccess,swimseasonlength,waterbodytype,watershedname,beach_upperlat,beach_upperlon,beach_lowerlat,beach_lowerlon,beach_county,beach_user_id,beach_status,countasbeach,agency_id,agency_code,agency_name,agency_jurisdiction,streetaddress,city,results_contact_person,zip,estmoncost,estcoastlineadvisorycost,esttotalcost,agency_comments,agency_county,agency_user_id,regional_board_number,regional_board_name
RB9,1644456,2010-07-20,2026-04-19T09:00:00.000Z,0,Total Coliforms,1,=,Equal to,17,MPN/100ml,25,9,9221-B,1,Results,null,San Diego,4,null,2010-07-20,null,null,null,null,null,null,null,null,null,null,10,10,1,703,23,100,SE-010,Fletcher Cove outlet,SE-010,32.991319,-117.274819,32.991319,-117.274819,Once a week,Once a week,Twice a week,SD,San Diego,Active,2019-02-01,2015-08-19,1,County of San Diego Department of Environmental Health,Solana Beach City Beaches,Solana Beach,UNKNOWN,1.36,0,0,Solana Beach,Pacific Ocean,Saltwater,No,CA505182,PUBLIC,12,Open Coast,San Diego River,32.99947,-117.277931,32.980308,-117.272381,San Diego,4,Active,1,23,SD,County of San Diego Department of Environmental Health,County,1255 Imperial Ave,San Diego,Keith Kezer,92112,>$250000,NULL,>$250000,Figures for sampling are yeaarly estimates for DEH only.,San Diego,1,9,San Diego
RB9,1644503,2010-07-19,2026-04-19T10:20:00.000Z,0,Total Coliforms,1,=,Equal to,5,MPN/100ml,13,9,9221-B,1,Results,null,San Diego,4,null,2010-07-19,null,null,null,null,null,null,null,null,null,null,10,10,1,404,23,101,EN-050,Cerezo Drive,EN-050,33.1285,-117.333806,33.1285,-117.333806,Once a week,Once a week,Twice a week,SD,San Diego,Active,null,2015-08-19,1,County of San Diego Department of Environmental Health,South Carlsbad State Beach,South Carlsbad State Beach,UNKNOWN,3.49,0,0,Carlsbad,Pacific Ocean,Saltwater,No,CA606869,PUBLIC,12,Open Coast,Escondido Creek-San Luis Rey River,33.1285,-117.3338,33.0815,-117.3115,San Diego,147,Active,1,23,SD,County of San Diego Department of Environmental Health,County,1255 Imperial Ave,San Diego,Keith Kezer,92112,>$250000,NULL,>$250000,Figures for sampling are yeaarly estimates for DEH only.,San Diego,1,9,San Diego
RB9,1644506,2010-07-19,2026-04-19T10:39:00.000Z,0,Total Coliforms,1,=,Equal to,1,MPN/100ml,13,9,9221-B,1,Results,null,San Diego,4,null,2010-07-19,null,null,null,null,null,null,null,null,null,null,10,10,1,403,23,101,EN-040,Palomar Airport,EN-040,33.119222,-117.327111,33.119222,-117.327111,Once a week,Once a week,Twice a week,SD,San Diego,Active,null,2015-08-19,1,County of San Diego Department of Environmental Health,South Carlsbad State Beach,South Carlsbad State Beach,UNKNOWN,3.49,0,0,Carlsbad,Pacific Ocean,Saltwater,No,CA606869,PUBLIC,12,Open Coast,Escondido Creek-San Luis Rey River,33.1285,-117.3338,33.0815,-117.3115,San Diego,147,Active,1,23,SD,County of San Diego Department of Environmental Health,County,1255 Imperial Ave,San Diego,Keith Kezer,92112,>$250000,NULL,>$250000,Figures for sampling are yeaarly estimates for DEH only.,San Diego,1,9,San Diego
RB9,1644512,2010-07-19,2026-04-19T11:00:00.000Z,0,Total Coliforms,1,=,Equal to,1,MPN/100ml,13,9,9221-B,1,Results,null,San Diego,4,null,2010-07-19,null,null,null,null,null,null,null,null,null,null,10,10,1

In [0]:
profiles = []
for t in ca_tables:
    df = spark.table(t)
    profiles.append(
        {
            "table_name": t,
            "row_count": df.count(),
            "column_count": len(df.columns),
            "columns": df.columns,
        }
    )

ca_beach_profiles_df = spark.createDataFrame(pd.DataFrame(profiles))
display(ca_beach_profiles_df)

ca_beach_profiles_df.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.ca_beach_table_profiles"
)

table_name,row_count,column_count,columns
toxictide.bronze.ca_beach_monitoring_stations_raw,1075,61,"List(regional_board, station_id, station_agency_id, beachname_id, station_name, station_description, agencystationidentifier, station_upperlat, station_upperlon, station_lowerlat, station_lowerlon, swimseason, offseason, asneededseason, countycode, countyname, status, updatedate, stations_insert_date, pointzero, datum, beach_agencyname, beach_name, description, beachtype, beach_length, attendancewinter, attendancesummer, nearestcityname, waterbodyname, waterbodyclass, ab411beach, usepaid, beachaccess, swimseasonlength, waterbodytype, watershedname, beach_upperlat, beach_upperlon, beach_lowerlat, beach_lowerlon, beach_county, beach_user_id, beach_status, countasbeach, agency_id, agency_code, agency_name, agency_jurisdiction, streetaddress, city, station_contact_person, zip, estmoncost, estcoastlineadvisorycost, esttotalcost, station_comments, county, agency_user_id, regional_board_number, regional_board_name)"
toxictide.bronze.ca_beach_postings_closures_raw,36765,98,"List(duration_calculated, regional_board, advisory_id, historicalstationname, prefix, mile, dateofadvisory, timeofadvisory, dateopened, timeopened, advisorytype, advisorycause, description, extent, startofextent, startofextenttype, endofextent, endofextenttype, source, advisory_comments, descriptionofarea, actionby, affectedlengthunit, substancename, substancevolume, substanceunit, additionalaction, actiondate, actiondescription, county, user_id, enterococcus, fecal_coliforms, total_coliforms, e_coli, ratio, advisory_record_create_date, station_id, station_agency_id, beachname_id, station_name, station_description, agencystationidentifier, station_upperlat, station_upperlon, station_lowerlat, station_lowerlon, swimseason, offseason, asneededseason, countycode, countyname, status, updatedate, advisories_insert_date, pointzero, datum, beachprofileid, beach_agencyname, beach_name, beachdescription, beachtype, beach_length, attendancewinter, attendancesummer, nearestcityname, waterbodyname, waterbodyclass, ab411beach, usepaid, beachaccess, swimseasonlength, waterbodytype, watershedname, beach_upperlat, beach_upperlon, beach_lowerlat, beach_lowerlon, beach_county, beach_user_id, beach_status, countasbeach, agency_id, agency_code, agency_name, agency_jurisdiction, streetaddress, city, advisories_contact_person, zip, estmoncost, estcoastlineadvisorycost, esttotalcost, agency_comments, agency_county, agency_user_id, regional_board_number, regional_board_name)"
toxictide.bronze.ca_beach_monitoring_results_raw,2392908,93,"List(regional_board, results_id, sampledate, starttime, starttimeind, parameter, qualifierid, qualifier, qualifierdescription, result, unit, labcode_id, analysismethodid, analysismethod, sampletypeid, sampletype, results_comments, county, user_id, upload_ts, result_record_create_date, labid, stormdrainflow, weather, tidalheight, surfheight, turbidity, watercolor, odor, distance, direction, rl, mdl, dilution, station_id, station_agency_id, beachname_id, station_name, station_description, agencystationidentifier, station_upperlat, station_upperlon, station_lowerlat, station_lowerlon, swimseason, offseason, asneededseason, countycode, countyname, status, updatedate, results_insert_date, pointzero, beach_agencyname, beach_name, description, beachtype, beach_length, attendancewinter, attendancesummer, nearestcityname, waterbodyname, waterbodyclass, ab411beach, usepaid, beachaccess, swimseasonlength, waterbodytype, watershedname, beach_upperlat, beach_upperlon, beach_lowerlat, beach_lowerlon, beach_county, beach_user_id, beach_status, countasbeach, agency_id, agency_code, agency_name, agency_jurisdiction, streetaddress, city, results_contact_person, zip, estmoncost, estcoastlineadvisorycost, esttotalcost, agency_comments, agency_county, agency_user_id, regional_board_number, regional_board_name)"


In [0]:
cce_inventory = (
    raw_inventory
    .filter(F.col("source_name") == "cce_moorings")
    .withColumn("source_group", F.regexp_extract("path", r"/raw/run_date=[^/]+/([^/]+)/", 1))
    .withColumn("filename", F.regexp_extract("path", r"/([^/]+)$", 1))
    .withColumn("sensor_family", F.regexp_extract("filename", r"_(?:D|P|M)_(.+)\.nc$", 1))
)

display(cce_inventory.orderBy(F.desc("length")))

source_name,path,length,modificationTime,source_group,filename,sensor_family
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_D_Meteorology-Wind.nc,84858112,2026-04-19T03:42:44.000Z,cce1,OS_CCE1_11_D_Meteorology-Wind.nc,Meteorology-Wind
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_11_D_Meteorology-Wind.nc,77846924,2026-04-19T08:16:53.000Z,cce2,OS_CCE2_11_D_Meteorology-Wind.nc,Meteorology-Wind
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_13_D_Meteorology-Wind.nc,64348520,2026-04-19T08:09:37.000Z,cce1,OS_CCE1_13_D_Meteorology-Wind.nc,Meteorology-Wind
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_10_D_Meteorology-Wind.nc,51802496,2026-04-19T08:17:16.000Z,cce2,OS_CCE2_10_D_Meteorology-Wind.nc,Meteorology-Wind
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_03_D_ADCP.nc,32240060,2026-04-19T08:11:42.000Z,cce2,OS_CCE2_03_D_ADCP.nc,ADCP
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_05_D_ADCP.nc,26062420,2026-04-19T03:37:52.000Z,cce1,OS_CCE1_05_D_ADCP.nc,ADCP
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_14_D_ADCP.nc,20878936,2026-04-19T03:44:27.000Z,cce1,OS_CCE1_14_D_ADCP.nc,ADCP
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_08_D_Meteorology-Wind.nc,17385008,2026-04-19T03:39:31.000Z,cce1,OS_CCE1_08_D_Meteorology-Wind.nc,Meteorology-Wind
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_13_D_ADCP.nc,17360008,2026-04-19T08:18:30.000Z,cce2,OS_CCE2_13_D_ADCP.nc,ADCP
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_07_D_Meteorology-Wind.nc,16963756,2026-04-19T03:38:27.000Z,cce1,OS_CCE1_07_D_Meteorology-Wind.nc,Meteorology-Wind


In [0]:
cce_family_counts = (
    cce_inventory
    .groupBy("source_group", "sensor_family")
    .count()
    .orderBy("source_group", "sensor_family")
)

display(cce_family_counts)

cce_family_counts.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.cce_moorings_family_counts"
)

source_group,sensor_family,count
cce1,ADCP,14
cce1,AQUADOPP,2
cce1,CHL,17
cce1,CTD,12
cce1,MICROCAT,4
cce1,MICROCAT-PART1,1
cce1,MICROCAT-PART2,1
cce1,MICROCAT-PART3,1
cce1,MICROCAT-PART4,1
cce1,MICROCAT-PART5,1


In [0]:
rep_families = ["CHL", "AQUADOPP", "CTD", "ADCP", "Meteorology", "Meteorology-Wind", "NO3", "OXYGEN"]

w = Window.partitionBy("source_group", "sensor_family").orderBy("length")

cce_representatives = (
    cce_inventory
    .filter(F.col("sensor_family").isin(rep_families))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("source_group", "sensor_family", "filename", "path", "length")
    .orderBy("source_group", "sensor_family")
)

display(cce_representatives)

cce_representatives.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.cce_moorings_representative_files"
)

source_group,sensor_family,filename,path,length
cce1,ADCP,OS_CCE1_04_D_ADCP.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_04_D_ADCP.nc,2606596
cce1,AQUADOPP,OS_CCE1_01_D_AQUADOPP.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_01_D_AQUADOPP.nc,358636
cce1,CHL,OS_CCE1_01_D_CHL.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_01_D_CHL.nc,8192
cce1,CTD,OS_CCE1_06_M_CTD.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_06_M_CTD.nc,1326288
cce1,Meteorology,OS_CCE1_04_D_Meteorology.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_04_D_Meteorology.nc,368296
cce1,Meteorology-Wind,OS_CCE1_09_D_Meteorology-Wind.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_09_D_Meteorology-Wind.nc,14084360
cce1,NO3,OS_CCE1_04_D_NO3.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_04_D_NO3.nc,25040
cce1,OXYGEN,OS_CCE1_16_D_OXYGEN.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_16_D_OXYGEN.nc,108160
cce2,ADCP,OS_CCE2_08_D_ADCP.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_08_D_ADCP.nc,2396072
cce2,AQUADOPP,OS_CCE2_01_D_AQUADOPP.nc,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_01_D_AQUADOPP.nc,1641360


In [0]:
def parse_netcdf_metadata(path: str, source_group: str, sensor_family: str):
    tmp_path = load_binary_to_tempfile(path, suffix=".nc")
    ds = xr.open_dataset(tmp_path, decode_times=False)

    file_rows = [{
        "path": path,
        "source_group": source_group,
        "sensor_family": sensor_family,
        "n_dims": len(ds.sizes),
        "n_coords": len(ds.coords),
        "n_data_vars": len(ds.data_vars),
        "global_attrs_json": json.dumps({k: str(v) for k, v in ds.attrs.items()}),
    }]

    var_rows = []
    for var_name, var in ds.variables.items():
        var_rows.append(
            {
                "path": path,
                "source_group": source_group,
                "sensor_family": sensor_family,
                "variable_name": var_name,
                "dims": json.dumps(list(var.dims)),
                "dtype": str(var.dtype),
                "attrs_json": json.dumps({k: str(v) for k, v in var.attrs.items()}),
            }
        )

    ds.close()
    os.remove(tmp_path)

    return pd.DataFrame(file_rows), pd.DataFrame(var_rows)


rep_rows = [r.asDict() for r in cce_representatives.collect()]

file_meta_parts = []
var_meta_parts = []

for r in rep_rows:
    file_meta_pdf, var_meta_pdf = parse_netcdf_metadata(
        path=r["path"],
        source_group=r["source_group"],
        sensor_family=r["sensor_family"],
    )
    file_meta_parts.append(file_meta_pdf)
    var_meta_parts.append(var_meta_pdf)

cce_rep_file_meta = pd.concat(file_meta_parts, ignore_index=True)
cce_rep_var_meta = pd.concat(var_meta_parts, ignore_index=True)

spark.createDataFrame(cce_rep_file_meta).write.mode("overwrite").saveAsTable(
    "toxictide.bronze.cce_representative_file_metadata"
)
spark.createDataFrame(cce_rep_var_meta).write.mode("overwrite").saveAsTable(
    "toxictide.bronze.cce_representative_variable_inventory"
)

display(spark.table("toxictide.bronze.cce_representative_file_metadata"))
display(spark.table("toxictide.bronze.cce_representative_variable_inventory"))

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-6904815970628430>, line 41
     38 var_meta_parts = []
     40 for r in rep_rows:
---> 41     file_meta_pdf, var_meta_pdf = parse_netcdf_metadata(
     42         path=r["path"],
     43         source_group=r["source_group"],
     44         sensor_family=r["sensor_family"],
     45     )
     46     file_meta_parts.append(file_meta_pdf)
     47     var_meta_parts.append(var_meta_pdf)

File <command-6904815970628430>, line 3, in parse_netcdf_metadata(path, source_group, sensor_family)
      1 def parse_netcdf_metadata(path: str, source_group: str, sensor_family: str):
      2     tmp_path = load_binary_to_tempfile(path, suffix=".nc")
----> 3     ds = xr.open_dataset(tmp_path, decode_times=False)
      5     file_rows = [{
      6         "path": path,
      7         "source_group": source_group,
   (...)
     12         

In [0]:
def parse_netcdf_datafile_to_pandas(path: str, source_group: str, sensor_family: str) -> pd.DataFrame:
    tmp_path = load_binary_to_tempfile(path, suffix=".nc")
    ds = xr.open_dataset(tmp_path, decode_times=False)

    pdf = ds.to_dataframe().reset_index()
    pdf["source_path"] = path
    pdf["source_group"] = source_group
    pdf["sensor_family"] = sensor_family

    ds.close()
    os.remove(tmp_path)

    pdf, _ = sanitize_pandas_columns(pdf)
    return pdf


small_families = ["CHL", "AQUADOPP"]

for fam in small_families:
    fam_rows = [
        r.asDict() for r in
        cce_inventory.filter(F.col("sensor_family") == fam).select("path", "source_group", "sensor_family").collect()
    ]

    first = True
    table_name = f"toxictide.bronze.cce_{sanitize_column_name(fam)}_raw"

    for r in fam_rows:
        pdf = parse_netcdf_datafile_to_pandas(
            path=r["path"],
            source_group=r["source_group"],
            sensor_family=r["sensor_family"],
        )
        write_pandas_to_table(pdf, table_name, mode="overwrite" if first else "append")
        first = False

    print(table_name, spark.table(table_name).count())

toxictide.bronze.cce_chl_raw 14545
toxictide.bronze.cce_aquadopp_raw 51009


# EasyArgo

In [0]:
argo_manifests = (
    spark.read
    .option("recursiveFileLookup", "true")
    .option("multiLine", "true")
    .json("s3://toxictide-raw/sources/easyoneargo/manifests/")
)

argo_manifest_records = (
    argo_manifests
    .select(
        "source_name",
        "run_id",
        "run_date",
        "status",
        F.explode("records").alias("record")
    )
    .select(
        "source_name",
        "run_id",
        "run_date",
        "status",
        F.col("record.filename").alias("filename"),
        F.col("record.local_cache_path").alias("local_cache_path"),
        F.col("record.s3_key").alias("s3_key"),
        F.col("record.size_bytes").alias("size_bytes"),
        F.col("record.source_url").alias("source_url"),
    )
)

display(argo_manifest_records)

argo_manifest_records.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.easyoneargo_manifest_records"
)

source_name,run_id,run_date,status,filename,local_cache_path,s3_key,size_bytes,source_url
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,127234.tar.gz,.cache\data_intake\easyoneargo\127234.tar.gz,sources/easyoneargo/raw/run_date=2026-04-19/127234.tar.gz,1595051070,https://www.seanoe.org/data/00961/107233/data/127234.tar.gz
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,126470.tar.gz,.cache\data_intake\easyoneargo\126470.tar.gz,sources/easyoneargo/raw/run_date=2026-04-19/126470.tar.gz,1586383338,https://www.seanoe.org/data/00961/107233/data/126470.tar.gz
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,argo_125529.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/125529.tar.gz
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,127233.tar.gz,.cache\data_intake\easyoneargo\127233.tar.gz,sources/easyoneargo/raw/run_date=2026-04-19/127233.tar.gz,7969262537,https://www.seanoe.org/data/00961/107233/data/127233.tar.gz
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,126471.tar.gz,.cache\data_intake\easyoneargo\126471.tar.gz,sources/easyoneargo/raw/run_date=2026-04-19/126471.tar.gz,7907834785,https://www.seanoe.org/data/00961/107233/data/126471.tar.gz
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,argo_126470.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/126470
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,argo_126471.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/126471
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,argo_127233.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/127233
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,argo_127234.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/127234
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,argo_125529.nc,null,null,null,https://www.seanoe.org/data/00961/107233/data/125529


In [0]:
argo_inventory = (
    raw_inventory
    .filter(F.col("source_name") == "easyoneargo")
)

display(argo_inventory.orderBy(F.desc("length")))

argo_inventory.write.mode("overwrite").saveAsTable(
    "toxictide.bronze.easyoneargo_archive_inventory"
)

source_name,path,length,modificationTime
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127233.tar.gz,7969262537,2026-04-19T08:00:49.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126471.tar.gz,7907834785,2026-04-19T08:00:34.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127234.tar.gz,1595051070,2026-04-19T07:50:26.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126470.tar.gz,1586383338,2026-04-19T07:50:47.000Z
